# Unit 13 exercises: evaluating the whole system (exposure, feedback loops, filter bubbles)

Run the recsys data generators before you begin. Your best recommender scores about **0.36 hit@10** on `val`. These exercises ask the questions that number cannot answer. **Who** gets recommended, and which books never are? Which **authors** get the exposure? Does a reader see a **narrower** slice of genres than they actually read? And what happens when readers start reading what the recommender suggests, and the recommender is **retrained on its own suggestions**? You will drill the three ideas of the lesson: **exposure fairness** (the Gini coefficient of exposure over the full catalog, author-level exposure next to each author's catalog share, the long-tail share, and the popularity-penalty trade-off curve), **feedback loops** (a short, seeded `simulate_feedback_loop` run, with its leakage guard), and **filter bubbles** (genre spread vs history). You will also read two **offline-metric limits** -- what a logged val split rewards, and which paths can serve a never-read book.

Read every number as a **descriptive measurement on our synthetic data**, not a verdict. An exposure Gini is a *proxy* for one kind of inequality, not "the" fairness. The feedback loop is a *stylized* exposure-to-consumption loop -- simulated readers take each shown book with a fixed probability -- and **not** a realistic model of what people like. The useful habit is to state a trade-off with numbers on both sides, not to preach.

Everything is measured on the **validation** split (`val`), on the scoreboard's **val cohort**: the 500 warm readers with at least one unseen val positive (cold readers excluded). Each reader's **seen** set is their train-positive history. The sealed `test` holdout stays sealed -- no cell here may touch it. Each answer is left for you to complete; no solutions live here.

A speed note. The content and neighbourhood paths fit in under a second. The Unit-8 two-tower takes about 15-25 s to train, so the setup cell fits it **exactly once** and every exercise reuses it (and the Unit-10 hybrid built on top of it). Do not fit a second two-tower, a matrix-factorization model or a feature tower anywhere in this notebook. A feedback-loop run refits item-item CF once per round, about 2 s a round, so a 3-round loop takes under 10 s. The core exercises run **one** loop (Exercise 4); the Challenge exercise 8 runs one more.

In [ ]:
import json
import math
from collections import Counter, defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    PopularityRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    catalog_coverage,
    head_ids_from_counts,
    head_share,
    rank,
)
from bookrec.fairness import (
    author_catalog_share,
    author_exposure_gini,
    exposure_counts,
    exposure_gini,
    genre_entropy,
    genre_spread_vs_history,
    gini,
    long_tail_share,
    popularity_penalized,
    simulate_feedback_loop,
    val_cohort,
)

SEED = 0        # the two-tower's training seed (Unit 8)
LOOP_SEED = 0   # the feedback-loop simulation's seed (one generator for every draw)
K = 10          # every readout is at the top-10 cutoff

# --- the real interaction log + content (the same substrate as Units 1-11) -----------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")        # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")     # {item_id: "space joined tokens"}
glove = bookrec.load_glove_subset()                                # the committed GloVe subset

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])      # readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
rows = pd.read_csv(interactions_path).to_dict("records")

# Train positives are each reader's `seen` set (their history); val positives are the target. We
# grade on `val`; no cell here may read the sealed held-out split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])

# The scoreboard's val cohort: warm readers with an unseen val positive (500 readers, ascending).
cohort = val_cohort(rows, catalog_ids, cold_readers)

# Popularity (Unit 2) also gives us each book's TRAIN-positive count -- warm books only.
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
train_counts = popularity.counts                              # {item_id: train-positive count}
head = set(head_ids_from_counts(train_counts, catalog_ids))   # the head: top 10% by train count
zero_train = set(catalog_ids) - set(train_counts)             # books with NO train positive

# The paths built in Units 3, 4, 7, 8 and 10. The two-tower is the ONE slow fit (~15-25 s).
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
hybrid = HybridRetrievalPath(lexical, two_tower).fit(rows, catalog=catalog_ids)   # pool 50, weight 0.7


def top_lists(path):
    # {reader: that reader's ranked top-K item ids} over the val cohort (seen books excluded).
    return {
        r: [c.item_id for c in rank(path.retrieve(r, {"seen": seen_by[r]}, K), n=K, exclude=seen_by[r])]
        for r in cohort
    }


def hit_of(lists):
    # hit@K of {reader: top-K ids}: the fraction of readers with an unseen val positive in their list.
    return float(np.mean([1.0 if set(top) & (val_pos_by[r] - seen_by[r]) else 0.0 for r, top in lists.items()]))

## Exercise 1 - The exposure Gini by hand, and why the never-shown books must count

**Exposure** is how many recommendation slots a book receives across all readers' top-10 lists. The **Gini coefficient** summarises how unequally exposure is spread: sort the per-book exposure counts ascending, $x_{(1)} \le \dots \le x_{(n)}$, and compute

$$G = \frac{\sum_{i=1}^{n} (2i - n - 1)\, x_{(i)}}{n \sum_i x_i}.$$

$GThis is the same number as the lesson's pairwise form $\sum_i \sum_j |x_i - x_j| / (2 n^2 \bar{x})$; the sorted form just avoids building an  \times n$ table.  = 0$ means every book gets the same exposure; $G = (n-1)/n$ means one book takes every slot. The one design decision that matters is **which books are in $n$**: the lesson's helper `exposure_gini(lists, catalog_ids)` counts the **full catalog**, so a book that is *never* recommended enters as a zero.

First, a toy you can check on paper: three readers' lists `{"A": [1, 2], "B": [1, 3], "C": [1, 2]}` over a five-book catalog `[1, 2, 3, 4, 5]`. Compute the full-catalog Gini by hand (expect 16/30, about 0.533) and the Gini over only the three books that were shown (expect 4/18, about 0.222). Then do it for real: build item-item's and the two-tower's top-10 lists with `top_lists`, count item-item's exposure per catalog book **by hand** (every catalog id present, zeros included), compute $G$ with the formula, and confirm it matches `exposure_gini` to floating-point precision (about 0.773). Finally compute the "shown-only" Gini for both paths (only books with exposure > 0). Expect the full-catalog Ginis about 0.773 (item-item) vs 0.929 (two-tower), but the shown-only Ginis only about 0.537 vs 0.598, over **980** vs **354** shown books. In two or three sentences, explain why a shown-only Gini is misleading, and why two paths' Ginis can only be compared over the **same** $n$.

In [ ]:
def gini_by_hand(values):
    # The formula above over a list of non-negative numbers (0.0 if they sum to 0).
    return None   # TODO: x = np.sort(np.array(values, dtype=float)); n = len(x);
                  #       return 0.0 if x.sum() == 0, else ((2 * np.arange(1, n + 1) - n - 1) * x).sum() / (n * x.sum())


toy_lists = {"A": [1, 2], "B": [1, 3], "C": [1, 2]}
toy_catalog = [1, 2, 3, 4, 5]
toy_counts = None        # TODO: {item: number of slots it fills in toy_lists} for EVERY item in toy_catalog (zeros too)
toy_full_gini = None     # TODO: gini_by_hand(list(toy_counts.values()))        -- 16/30 ~0.533
toy_shown_gini = None    # TODO: gini_by_hand([c for c in toy_counts.values() if c > 0])   -- 4/18 ~0.222
toy_matches_helper = None  # TODO: bool - abs(toy_full_gini - exposure_gini(toy_lists, toy_catalog)) < 1e-12

item_item_lists = None   # TODO: top_lists(item_item)
two_tower_lists = None   # TODO: top_lists(two_tower)

item_item_exposure = None  # TODO: {item_id: 0 for item_id in catalog_ids}, then +1 per slot in item_item_lists
item_item_gini_hand = None    # TODO: gini_by_hand(list(item_item_exposure.values()))   -- ~0.773
item_item_gini = None         # TODO: exposure_gini(item_item_lists, catalog_ids)
hand_matches_helper = None    # TODO: bool - abs(item_item_gini_hand - item_item_gini) < 1e-12
two_tower_gini = None         # TODO: exposure_gini(two_tower_lists, catalog_ids)        -- ~0.929

item_item_shown = None        # TODO: [c for c in item_item_exposure.values() if c > 0]   -- 980 books
two_tower_shown = None        # TODO: the same for two_tower_lists (exposure_counts(two_tower_lists, catalog_ids)
                              #       gives the full per-book counts)                       -- 354 books
item_item_shown_gini = None   # TODO: gini(item_item_shown)   -- ~0.537
two_tower_shown_gini = None   # TODO: gini(two_tower_shown)   -- ~0.598

full_catalog_explanation = "TODO"  # 2-3 sentences: a shown-only Gini drops the never-recommended books -- the
                                   # most unequal part of the picture -- so it flatters a narrow path; and it
                                   # measures each path over a DIFFERENT n (980 vs 354 books), so the two
                                   # numbers are not comparable; counting the full catalog fixes n for every path

## Exercise 2 - The whole-system scoreboard: accuracy next to exposure, books and authors

Put six paths on **one** scoreboard: popularity (Unit 2), lexical BM25 (Unit 3), semantic GloVe (Unit 7), item-item CF (Unit 4), the two-tower (Unit 8) and the Unit-10 hybrid. For each, build its cohort top-10 lists (reuse Exercise 1's two) and record hit@10 (`hit_of`), catalog coverage, head-share (`head_share(lists, head)`), long-tail share (`long_tail_share(lists, head)`, the fraction of slots that go to non-head books), the item exposure Gini, and the **author** exposure Gini (`author_exposure_gini(lists, catalog)`) in one pandas `DataFrame`, sorted by hit@10 (highest first).

Expect (hit / coverage / item Gini / author Gini): hybrid ~0.362 / 0.192 / 0.922 / 0.677, two-tower ~0.340 / 0.177 / 0.929 / 0.687, item-item ~0.252 / 0.490 / 0.773 / 0.474, lexical ~0.158 / 0.278 / 0.888 / 0.600, popularity ~0.108 / 0.008 / 0.995 / 0.970, semantic ~0.102 / 0.354 / 0.857 / 0.573. Confirm that the two most accurate paths are among the most concentrated (item Gini above 0.9), and that item-item is the least concentrated path of all.

An author with more books *should* get more exposure, so author inequality is partly the catalog's own. `author_catalog_share(catalog)` gives each author's share of the 2,000 books (300 authors). Compute the Gini of those shares -- the author Gini you would see if exposure exactly followed books-per-author (expect ~0.211) -- and confirm every path sits above it. Then, for popularity and item-item, find the **most-exposed author** and compare their share of all exposure slots with their share of the catalog (expect popularity's top author to take ~18.9% of the slots with only ~0.5% of the books, and item-item's ~1.7% with ~0.4%). In two or three sentences, say what the author Gini adds to the item Gini, and why you report it next to the catalog-share baseline instead of reading it alone.

In [ ]:
paths = {
    "popularity": popularity,
    "lexical": lexical,
    "semantic": semantic,
    "item-item": item_item,
    "two-tower": two_tower,
    "hybrid": hybrid,
}
lists_by_path = None   # TODO: {name: top_lists(path) for name, path in paths.items()}
                       #       (reuse item_item_lists / two_tower_lists instead of rebuilding them)

scoreboard = None      # TODO: pd.DataFrame([{"path": name, "hit@10": hit_of(lists),
                       #           "coverage": catalog_coverage(list(lists.values()), catalog_ids),
                       #           "head_share": head_share(list(lists.values()), head),
                       #           "long_tail_share": long_tail_share(lists, head),
                       #           "item_gini": exposure_gini(lists, catalog_ids),
                       #           "author_gini": author_exposure_gini(lists, catalog)}
                       #       for name, lists in lists_by_path.items()]).sort_values("hit@10", ascending=False)

accurate_are_concentrated = None   # TODO: bool - the item_gini of "two-tower" and "hybrid" are both > 0.9
item_item_least_concentrated = None  # TODO: bool - "item-item" has the smallest item_gini in the table

author_share = None          # TODO: author_catalog_share(catalog)   -- {author_id: share of the 2000 books}
catalog_share_gini = None    # TODO: gini(author_share.values())      -- ~0.211 (the books-per-author baseline)
all_above_baseline = None    # TODO: bool - every path's author_gini > catalog_share_gini


def top_author(lists):
    # (author_id, share of all exposure slots) for the most-exposed author in `lists`.
    return None   # TODO: count catalog[item].fields["author_id"] over every slot (a Counter);
                  #       take .most_common(1)[0] and divide its count by the total number of slots


popularity_top_author = None   # TODO: top_author(lists_by_path["popularity"])   -- share ~0.189
item_item_top_author = None    # TODO: top_author(lists_by_path["item-item"])    -- share ~0.017
# Compare each with author_share[author_id] (~0.005 and ~0.004).

author_explanation = "TODO"  # 2-3 sentences: the author Gini asks whether exposure concentrates on a few
                             # CREATORS (popularity gives one author ~19% of the slots with 0.5% of the books);
                             # but authors own different numbers of books, so even exposure-per-book would
                             # give an author Gini of ~0.21 -- read every path against that baseline

## Exercise 3 - A three-point popularity-penalty curve: what does fairer exposure cost?

The lesson's mitigation for static concentration is a **popularity-penalised re-rank**. `popularity_penalized(path, penalty, train_counts, pool=50)` asks the inner path for a pool of 50 candidates, min-max scales their scores to $[0, 1]$, and re-scores each candidate as

$$\text{score}' = \text{minmax}(\text{score}) - \lambda \cdot \frac{\log(1 + \text{count})}{\log(1 + \text{max count})},$$

where `count` is the book's train-positive count (0 for a zero-train book) and `max count` is the largest count. A penalty of $\lambda = 0$ keeps the inner order.

First, check the penalty term by hand: what does it subtract for the single most-read book, and for a zero-train book, at $\lambda = 0.3$? (Expect exactly 0.3 and exactly 0.) Then wrap item-item at three points of the lesson's grid, $\lambda \in \{0, 0.3, 1.0\}$, and record hit@10, item Gini, coverage and head-share for each. Confirm $\lambda = 0$ reproduces plain item-item's lists exactly, that the Gini **strictly decreases** along the three points (expect ~0.773 -> 0.747 -> 0.713), and that hit@10 **never increases** (expect ~0.252 -> 0.224 -> 0.112). Record the signed deltas against $\lambda = 0$ (expect hit about -0.028 and -0.140; Gini about -0.026 and -0.060). In three or four sentences, read the curve as a **trade-off** (a Pareto readout, not a winner): what does $\lambda = 0.3$ buy and cost compared with $\lambda = 1.0$, and who should decide which point a real service uses?

In [ ]:
LAMBDAS = [0.0, 0.3, 1.0]   # three points of the lesson's predeclared grid {0, 0.1, 0.3, 0.5, 1.0}
max_count = None            # TODO: max(train_counts.values())


def penalty_term(item_id, lam):
    # What the re-rank subtracts from a candidate's min-max score.
    return None   # TODO: lam * math.log1p(train_counts.get(item_id, 0)) / math.log1p(max_count)


most_read_book = None        # TODO: the item id with the largest train count
most_read_penalty = None     # TODO: penalty_term(most_read_book, 0.3)            -- exactly 0.3
zero_train_penalty = None    # TODO: penalty_term(min(zero_train), 0.3)          -- exactly 0.0

penalty_curve = None   # TODO: a DataFrame with one row per lam in LAMBDAS: lists = top_lists(
                       #       popularity_penalized(item_item, lam, train_counts, pool=50)); columns "lambda",
                       #       "hit@10", "item_gini", "coverage", "head_share" (computed as in Exercise 2)
lambda0_reproduces = None    # TODO: bool - the lam = 0 lists == item_item_lists (Exercise 1), reader for reader

gini_strictly_falls = None   # TODO: bool - each item_gini < the previous one          (~0.773, 0.747, 0.713)
hit_never_rises = None       # TODO: bool - each hit@10 <= the previous one             (~0.252, 0.224, 0.112)
hit_deltas = None            # TODO: [hit@10 at lam - hit@10 at 0 for lam in (0.3, 1.0)]    (~ -0.028, -0.140)
gini_deltas = None           # TODO: [item_gini at lam - item_gini at 0 for lam in (0.3, 1.0)]  (~ -0.026, -0.060)

tradeoff_explanation = "TODO"  # 3-4 sentences: lam = 0.3 buys a modest Gini drop (and a big head-share drop)
                               # for a small hit cost; lam = 1.0 roughly doubles the Gini drop but halves hit@10;
                               # there is no free point -- the curve is a menu, and choosing a point is a
                               # value judgment (readers? authors? the business?), not a metric's verdict

## Exercise 4 - A short feedback loop: retraining on your own recommendations

A recommender's training log is not handed down from nature: readers mostly read what they were *shown*, and tomorrow's model is trained on today's reads. `simulate_feedback_loop` runs that loop on our data. Each round it fits a fresh path on the (growing) train log; shows every cohort reader a top-10 that excludes everything they have read so far; lets each shown book be "read" independently with probability `accept_prob` (0.3, so about 3 reads per reader per round); appends those reads to the train log as new train rows; and records the round's readouts -- coverage, head-share, exposure Gini and mean genre entropy of the shown lists, and a **clean hit@10**.

Two pinned rules make the readouts honest. The **leakage guard**: a simulated reader never "reads" one of their own held-out (val or sealed) positives, so the simulation never hands the model a held-out answer. And **clean hit** scores each round's refit path on the *original* val positives with the reader's *original* train history as the seen set -- so a growing seen set cannot move the accuracy readout for a bookkeeping reason.

Before you run it, **predict**: after three rounds, will item-item's coverage rise or fall, and will its clean hit rise or fall? Write your prediction first. Then run a 3-round loop on item-item (`simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog, seed=LOOP_SEED, rounds=3)` -- note it takes the path **class**, a factory it calls once per round) and put the rounds in a `DataFrame` (columns `round`, `coverage`, `head_share`, `gini`, `genre_entropy`, `clean_hit`, `accepted`). Expect coverage ~0.490 -> 0.412, Gini ~0.773 -> 0.835, clean hit ~0.252 -> 0.266 -> 0.256 -> 0.206, and about 1,300-1,430 reads accepted per round. Confirm round 0 reproduces Exercise 1's item-item Gini and Exercise 2's item-item coverage and hit. Then **audit the guard** yourself from each round's `new_rows`: every simulated row is a `train` row with `label` 1, no reader ever "reads" one of their own val positives, and no (reader, book) pair is read twice. In three or four sentences, explain what the loop does to exposure and why, why clean hit eventually falls (the model learns the *recommender*, not the readers), and what would go wrong without the guard. (The lesson measured the leak: in round 1 the guard blocks 79 reads, and a refit on the *unguarded* log scores 0.288 clean hit against the guarded 0.266 -- the model was simply trained on some of the answers it is graded on.)

In [ ]:
loop_prediction = "TODO"   # write this BEFORE running the loop: coverage up or down? clean hit up or down?

loop = None           # TODO: simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog,
                      #                              seed=LOOP_SEED, rounds=3)     -- 4 RoundMetrics (rounds 0..3)
loop_table = None     # TODO: pd.DataFrame([{"round": m.round, "coverage": m.coverage, "head_share": m.head_share,
                      #           "gini": m.gini, "genre_entropy": m.genre_entropy, "clean_hit": m.clean_hit,
                      #           "accepted": m.accepted} for m in loop])

round0_matches = None     # TODO: bool - loop[0].gini ~ item_item_gini, loop[0].coverage ~ the item-item coverage
                          #       and loop[0].clean_hit ~ hit_of(item_item_lists) (all within 1e-9)
coverage_fell = None      # TODO: loop[0].coverage - loop[-1].coverage    -- ~ +0.078 (0.490 -> 0.412)
gini_rose = None          # TODO: loop[-1].gini - loop[0].gini            -- ~ +0.062 (0.773 -> 0.835)
clean_hit_change = None   # TODO: loop[-1].clean_hit - loop[0].clean_hit  -- ~ -0.046 (0.252 -> 0.206)

simulated_rows = None     # TODO: every row in m.new_rows for every m in loop (a flat list of dicts)
all_train_rows = None     # TODO: bool - every simulated row has split "train" and label 1
guard_holds = None        # TODO: bool - no simulated row's item_id is in val_pos_by[its reader_id]
no_double_reads = None    # TODO: bool - the (reader_id, item_id) pairs of simulated_rows are all distinct

loop_explanation = "TODO"  # 3-4 sentences: each round the model is retrained on reads it caused, so the books it
                           # already favoured gain counts and co-occurrences and get shown even more (coverage
                           # falls, Gini rises -- "rich get richer"); clean hit eventually falls because the new
                           # rows teach CF what the recommender showed, not what readers want; without the guard,
                           # readers would "read" their own val books and clean hit would rise -- a leak, not learning

## Exercise 5 - Filter bubbles: genre spread of the recommendations vs the reader's history

A **filter bubble** is a recommender showing a reader a *narrower* slice than they actually read. One measurable proxy is **genre entropy**: `genre_entropy(items, catalog)` is the Shannon entropy, in bits, of the genre labels across a list of books (each book contributes each of its genres). `genre_spread_vs_history(lists, history, catalog)` returns, per reader, the pair *(entropy of the recommendations, entropy of the reader's own history)*.

For item-item, the two-tower and lexical, compute each cohort reader's pair using their train history (`seen_by`) as the history. Report the mean recommendation entropy, the mean history entropy (the same for every path, ~1.564 bits), and the fraction of readers whose recommendations are **narrower** than their history. Expect item-item ~1.640 bits with ~42% of readers narrower; the two-tower ~0.681 bits with ~84% narrower; lexical ~0.300 bits with ~89% narrower. Then read **narrowing over time** from Exercise 4's loop: the `genre_entropy` column falls from ~1.640 at round 0 to ~1.184 at round 3. In three or four sentences, explain why the most accurate path (two-tower) and the content path (lexical) are the narrowest, what the feedback loop adds on top of a static bubble, and one honest limit of genre entropy as a bubble measure.

In [ ]:
history = None        # TODO: {r: seen_by[r] for r in cohort}   -- each reader's train history

spread = None         # TODO: {name: genre_spread_vs_history(lists_by_path[name], history, catalog)
                      #        for name in ("item-item", "two-tower", "lexical")}
bubble_table = None   # TODO: pd.DataFrame with one row per path: "path", "mean_rec_entropy" (mean of the first
                      #       items of the pairs), "mean_history_entropy" (mean of the second items),
                      #       "frac_narrower" (fraction of readers with rec entropy < history entropy)
                      #       -- item-item ~1.640 / 1.564 / 0.42, two-tower ~0.681 / 1.564 / 0.84,
                      #          lexical ~0.300 / 1.564 / 0.89

loop_entropy = None        # TODO: [m.genre_entropy for m in loop]   (Exercise 4)   -- ~1.640 ... ~1.184
loop_narrowing = None      # TODO: loop_entropy[0] - min(loop_entropy[1:])        -- ~ +0.46 bits

bubble_explanation = "TODO"  # 3-4 sentences: the two-tower packs each list with near neighbours of one taste
                             # region and lexical with text matches of the same topic, so both are narrower than
                             # what readers actually read; the loop then narrows item-item further because the
                             # reads it causes reinforce the genres it already showed; genre entropy only sees
                             # genre LABELS (not authors, styles or eras), so it is one proxy, not the bubble itself

## Exercise 6 - An offline-metric limit: the val split reflects the logging policy

Every hit@10 in this course is computed on a *logged* val split -- reads that happened under whatever the readers were shown at the time. A log built that way is not a neutral sample of what readers would like: it leans toward books that were already visible. Measure two things on the val cohort.

First, the share of val **targets** (each cohort reader's unseen val positives, pooled) that are **head** books (the top 10% by train count). Expect about **44.1%** -- for a head that holds only 10% of the catalog. Second, split the readers by target type: a **head-target hit** is a hit on a head val book (among readers who have at least one), a **tail-target hit** is a hit on a non-head val book (among readers who have at least one). Measure both for item-item and the two-tower. Expect item-item ~0.356 head vs ~0.032 tail (about 11x, over 323 and 370 readers), and the two-tower ~0.517 head vs ~0.014 tail (about 38x). Confirm the head-target hit is at least 5x the tail-target hit for both. In three or four sentences, explain why an offline score built on such a log rewards concentration, and why this is a measured **association**, not proof that the metric is *inflated* by a precise amount.

In [ ]:
val_targets = None           # TODO: [item for r in cohort for item in val_pos_by[r] - seen_by[r]]
head_share_of_targets = None # TODO: float(np.mean([item in head for item in val_targets]))   -- ~0.441


def head_tail_hits(lists):
    # (head-target hit, tail-target hit, readers with a head target, readers with a tail target).
    return None   # TODO: for each (r, top) in lists.items(): relevant = val_pos_by[r] - seen_by[r];
                  #       if relevant & head: record 1.0 if set(top) & relevant & head else 0.0 (head list);
                  #       if relevant - head: record 1.0 if set(top) & (relevant - head) else 0.0 (tail list);
                  #       return the two means and the two list lengths


item_item_head_tail = None   # TODO: head_tail_hits(lists_by_path["item-item"])   -- ~0.356 / ~0.032 / 323 / 370
two_tower_head_tail = None   # TODO: head_tail_hits(lists_by_path["two-tower"])   -- ~0.517 / ~0.014 / 323 / 370
head_dominates = None        # TODO: bool - head hit >= 5 * tail hit for both paths

logging_explanation = "TODO"  # 3-4 sentences: 44% of val targets sit in a 10% head because the log was
                              # produced while those books were visible; a path that recommends the head hits
                              # those targets, so offline hit@10 rewards concentration; but we only measured an
                              # association -- we have no log from a different policy, so we cannot say how much
                              # of the score is "inflation" versus real preference for those books

## Exercise 7 - The cold-start thread closes: which paths surface a never-read book?

The cold-start thread ran through Unit 6 (cold readers), Unit 7 (content paths) and Unit 9 (the feature tower). There are **818 zero-train books** -- books with no train positive at all (`zero_train`, built from train positives only). For each of the six paths in Exercise 2, count how many **distinct** zero-train books appear anywhere in the cohort's top-10 lists. Expect exactly **0** for popularity, item-item and the two-tower; **128** for lexical, **263** for semantic, and **17** for the hybrid.

You will **not** fit the feature tower here (it would cost another ~25-35 s). The lesson fit it on this same 500-reader cohort: it surfaces **109** of the 818 -- the only *learned/collaborative* path that reaches cold books (Unit 9 measured 113 on its own reader set), while the ID-only two-tower stays at 0. Use that cited number in your answer. In three or four sentences, explain why the warm-only collaborative paths can never surface a zero-train book, where the hybrid's 17 come from, and what this means for an author whose first book has just been added to the catalog.

In [ ]:
zero_train_surfaced = None   # TODO: {name: len({item for top in lists.values() for item in top} & zero_train)
                             #        for name, lists in lists_by_path.items()}
                             #   -- popularity 0, item-item 0, two-tower 0, lexical 128, semantic 263, hybrid 17
warm_only_zero = None        # TODO: bool - popularity, item-item and two-tower all surface exactly 0
content_paths_reach = None   # TODO: bool - lexical and semantic both surface more than 0

CITED_FEATURE_TOWER_COLD = 109   # the lesson: zero-train books the feature tower surfaces on this cohort (of 818)

cold_explanation = "TODO"  # 3-4 sentences: popularity, item-item and the ID-only two-tower learn only from train
                           # interactions, so a book with none has no count, no co-reads and only negative
                           # gradients -- it can never score; content paths score text, so they reach cold books;
                           # the hybrid inherits its 17 from its lexical half; a debut author gets zero exposure
                           # from the collaborative paths until someone finds the book another way

## Exercise 8 - Challenge: an exploration slot inside the feedback loop

The lesson's primary mitigation for a feedback loop is an **exploration slot**: in every round, the **last** of each reader's 10 shown slots is replaced by a seeded random warm book the reader has not read and that is not already in the list. `simulate_feedback_loop(..., mitigation="explore")` does exactly that (same seed, same cohort, same guard).

Run a second 3-round item-item loop with `mitigation="explore"` and compare it with Exercise 4's loop at round 3. Expect coverage ~0.455 vs ~0.412 (about +0.043), Gini ~0.812 vs ~0.835 (about -0.023), and genre entropy ~1.528 vs ~1.184. Record the accuracy cost as two **signed** deltas, mitigated minus unmitigated: `cost_0` at round 0 (expect exactly 0 -- clean hit scores the refit, *unmitigated* path, and round 0 fits the same original log) and `cost_R` at round 3 (expect about +0.010: 0.216 vs 0.206). In three or four sentences, explain what one random slot in ten buys, why it does not *stop* the concentration, and why `cost_R` can come out positive here. (Optional extension: rerun both loops with `rounds=6` and see whether the gap holds -- the lesson's longer runs say it does. It costs about 30 s more.)

In [ ]:
explore_loop = None   # TODO: simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog,
                      #                              seed=LOOP_SEED, rounds=3, mitigation="explore")

coverage_gain = None   # TODO: explore_loop[-1].coverage - loop[-1].coverage   -- ~ +0.043
gini_drop = None       # TODO: explore_loop[-1].gini - loop[-1].gini           -- ~ -0.023
entropy_gain = None    # TODO: explore_loop[-1].genre_entropy - loop[-1].genre_entropy   -- ~ +0.34
cost_0 = None          # TODO: explore_loop[0].clean_hit - loop[0].clean_hit   -- exactly 0.0
cost_R = None          # TODO: explore_loop[-1].clean_hit - loop[-1].clean_hit -- ~ +0.010
explore_helps = None   # TODO: bool - coverage_gain >= 0.04 and gini_drop <= -0.01

explore_explanation = "TODO"  # 3-4 sentences: the slot puts books the model would never show in front of readers,
                              # and the reads it causes enter the log, so the next refit has evidence about them;
                              # nine slots still follow the model, so concentration slows rather than stops;
                              # cost_R can be positive because a more varied training log hurts CF's clean hit
                              # less than a self-reinforced one -- the cost is not guaranteed to be negative

## Exercise 9 - Challenge: a calibrated genre re-rank against the filter bubble

Exercise 5 showed the two-tower's lists are much narrower than readers' histories. A different mitigation from the popularity penalty is a **calibrated re-rank**: choose the 10 books so that the *genre mix of the list* resembles the *genre mix of the reader's history*. Measure the mismatch with the Kullback-Leibler divergence $\mathrm{KL}(p \,\|\, \tilde q) = \sum_g p_g \log_2 (p_g / \tilde q_g)$, where $p$ is the history's genre distribution, $q$ the list's, and $\tilde q = 0.99\,q + 0.01\,p$ (the small mix keeps a missing genre from dividing by zero). Then build each list **greedily**: from the two-tower's pool of 50 candidates (scores min-max scaled to $[0, 1]$), repeatedly add the candidate that maximises

$$(1 - \beta)\cdot\text{score} - \beta \cdot \mathrm{KL}(p \,\|\, \tilde q(\text{list} + \text{candidate})),$$

breaking ties toward the smaller item id, until the list holds 10 books.

The scaffold gives `genre_distribution` and `kl_to_history`. Write `calibrated_top(reader, beta)` and evaluate $\beta = 0$ and $\beta = 0.5$ over the cohort. Confirm $\beta = 0$ reproduces the two-tower's own lists exactly. At $\beta = 0.5$ expect hit@10 ~0.340 -> ~0.332, mean genre entropy ~0.681 -> ~1.219 bits, mean KL to history ~1.688 -> ~0.548, coverage ~0.177 -> ~0.230 and item Gini ~0.929 -> ~0.913. In three or four sentences, compare this mitigation with the popularity penalty of Exercise 3: which inequality does each target, and what does each cost?

In [ ]:
def genre_distribution(items):
    # Given: {genre: share} over the genre labels of `items` (each book contributes each of its genres).
    counts = Counter(g for item in items for g in catalog[item].fields["genres"].split(";") if g)
    total = sum(counts.values())
    return {g: c / total for g, c in counts.items()} if total else {}


def kl_to_history(p, items, alpha=0.01):
    # Given: KL(p || q~) in bits, q~ = (1 - alpha) * q(items) + alpha * p.
    q = genre_distribution(items)
    return sum(pg * math.log(pg / ((1 - alpha) * q.get(g, 0.0) + alpha * pg), 2) for g, pg in p.items())


def calibrated_top(reader, beta, pool=50):
    # Greedy calibrated re-rank of the two-tower's pool for one reader (a list of K item ids).
    return None   # TODO: candidates = two_tower.retrieve(reader, {"seen": seen_by[reader]}, pool);
                  #       min-max scale their scores to [0, 1] (all 1.0 if every score is equal) in a dict;
                  #       p = genre_distribution(seen_by[reader]); chosen = [];
                  #       while len(chosen) < K and candidates remain: pick, over the remaining ids in
                  #       ascending order, the max of (1 - beta) * score[i] - beta * kl_to_history(p, chosen + [i]);
                  #       append it to chosen and remove it from the remaining candidates


calibrated_lists = None    # TODO: {beta: {r: calibrated_top(r, beta) for r in cohort} for beta in (0.0, 0.5)}
beta0_reproduces = None    # TODO: bool - calibrated_lists[0.0] == lists_by_path["two-tower"]

calibration_table = None   # TODO: a DataFrame, one row per beta: "beta", "hit@10", "mean_genre_entropy",
                           #       "mean_kl_to_history" (kl_to_history(genre_distribution(seen_by[r]), top)
                           #       averaged over readers), "coverage", "item_gini"
                           #   -- beta 0: ~0.340 / 0.681 / 1.688 / 0.177 / 0.929
                           #      beta 0.5: ~0.332 / 1.219 / 0.548 / 0.230 / 0.913

calibration_explanation = "TODO"  # 3-4 sentences: calibration targets the PER-READER bubble (the list matches the
                                  # reader's own genre mix) at a small hit cost; the popularity penalty targets
                                  # CATALOG-WIDE concentration at a larger hit cost; calibration also widens coverage
                                  # a little, but neither is "the" fix -- they measure and trade different things

## Exercise 10 - Challenge: a reader-side slice -- light vs heavy readers

Exposure fairness looks at the *items* side. The other side is the **readers**: does the recommender serve some groups of readers better than others? Split the cohort at the **median** train-history length (8 books): **light** readers have at most 8 train books (273 readers), **heavy** readers more than 8 (227 readers). For popularity, item-item and the two-tower, report each group's hit@10 and head-share (reuse `lists_by_path`; `hit_of` works on any sub-dict). Expect hit@10 light vs heavy: popularity ~0.121 vs ~0.093, item-item ~0.249 vs ~0.256, two-tower ~0.374 vs ~0.300; and item-item's head-share ~0.443 (light) vs ~0.549 (heavy).

A gap between two groups needs an error bar before it means anything. The groups hold **different** readers, so use the **unpaired** standard error of the difference, $\sqrt{s_1^2/n_1 + s_2^2/n_2}$, where $s^2$ is each group's per-reader hit standard deviation (`np.std(hits, ddof=1)`, as in Unit 12's paired standard error), squared. Expect the two-tower's light-minus-heavy gap ~+0.074 with a standard error of ~0.042 (under 2 SE). Do **not** compare the two groups' *coverage*: they hold different numbers of readers, so their lists have different numbers of slots -- the same different-$n$ trap as Exercise 1. In three or four sentences, report what you can and cannot conclude about the two-tower serving light readers better, and why item-item's heavy readers see more of the head.

In [ ]:
history_length = None   # TODO: {r: len(seen_by[r]) for r in cohort}
median_length = None    # TODO: float(np.median(list(history_length.values())))   -- 8.0
light = None            # TODO: [r for r in cohort if history_length[r] <= median_length]   -- 273 readers
heavy = None            # TODO: [r for r in cohort if history_length[r] > median_length]    -- 227 readers

group_table = None      # TODO: a DataFrame, one row per (path, group) for popularity / item-item / two-tower:
                        #       "path", "group", "readers", "hit@10" (hit_of on the group's sub-dict of
                        #       lists_by_path[path]), "head_share" (head_share of those lists)


def per_reader_hits(lists, readers):
    # 1.0 / 0.0 per reader: did their list hit an unseen val positive?
    return None   # TODO: np.array([1.0 if set(lists[r]) & (val_pos_by[r] - seen_by[r]) else 0.0 for r in readers])


tt_light = None    # TODO: per_reader_hits(lists_by_path["two-tower"], light)
tt_heavy = None    # TODO: per_reader_hits(lists_by_path["two-tower"], heavy)
tt_gap = None      # TODO: tt_light.mean() - tt_heavy.mean()                                    -- ~ +0.074
tt_gap_se = None   # TODO: math.sqrt(np.std(tt_light, ddof=1) ** 2 / len(tt_light)
                   #                  + np.std(tt_heavy, ddof=1) ** 2 / len(tt_heavy))
                   #                                                                            -- ~0.042
gap_is_clear = None   # TODO: bool - abs(tt_gap) >= 2 * tt_gap_se   (record honestly; expect False)

reader_side_explanation = "TODO"  # 3-4 sentences: the two-tower's light readers score higher in this sample, but
                                  # the gap is under 2 standard errors, so it is a hint, not a finding; the slice
                                  # itself is the point -- an average hit can hide a group served worse; heavy
                                  # readers give item-item more co-read evidence about popular books, so their
                                  # lists lean further toward the head